In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# ── Configs ───────────────────────────────────────────────────────────────────
IMG_SIZE = 224
CHANNELS = 3
FRAME_COUNT = 20
THRESHOLD = 0.8
MODEL_PATH = "models/cctvmodel_advanced.keras"

# ── Load Models ───────────────────────────────────────────────────────────────
print("Loading models...")
extractor = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
)
model = tf.keras.models.load_model(MODEL_PATH)
print("Ready. Press A to toggle analysis. Press Q to quit.\n")

# ── Webcam ────────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

# ── Button ────────────────────────────────────────────────────────────────────
BTN_X, BTN_Y = 20, 20
BTN_W, BTN_H = 160, 45


def draw_button(frame, active):
    label = "Stop Analysis" if active else "Start Analysis"
    color = (0, 0, 180) if active else (0, 140, 0)
    cv2.rectangle(frame, (BTN_X, BTN_Y), (BTN_X + BTN_W, BTN_Y + BTN_H), color, -1)
    cv2.rectangle(
        frame, (BTN_X, BTN_Y), (BTN_X + BTN_W, BTN_Y + BTN_H), (255, 255, 255), 1
    )
    cv2.putText(
        frame,
        label,
        (BTN_X + 10, BTN_Y + 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )


def is_button_click(x, y):
    return BTN_X <= x <= BTN_X + BTN_W and BTN_Y <= y <= BTN_Y + BTN_H


def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        if is_button_click(x, y):
            param["clicked"] = True


# ── State ─────────────────────────────────────────────────────────────────────
buffer = []
label = ""
confidence = 0.0
color = (200, 200, 200)
analyzing = False
mouse_state = {"clicked": False}

cv2.namedWindow("Violence Detection — Real Time")
cv2.setMouseCallback("Violence Detection — Real Time", on_mouse, mouse_state)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Camera not accessible.")
        break

    display = frame.copy()
    h, w = display.shape[:2]

    # ── Handle button click ───────────────────────────────────────────────────
    if mouse_state["clicked"]:
        analyzing = not analyzing
        mouse_state["clicked"] = False
        buffer = []
        if not analyzing:
            label = ""
            confidence = 0.0

    # ── Analysis ──────────────────────────────────────────────────────────────
    if analyzing:
        buffer.append(frame.copy())

        if len(buffer) == FRAME_COUNT:
            processed = np.array(
                [
                    preprocess_input(
                        cv2.resize(f, (IMG_SIZE, IMG_SIZE)).astype(np.float32)
                    )
                    for f in buffer
                ]
            )
            features = extractor.predict(processed, verbose=0)
            prob = model.predict(np.expand_dims(features, axis=0), verbose=0)[0][0]
            confidence = prob * 100
            label = "Violence" if prob >= THRESHOLD else "Normal"
            color = (0, 0, 220) if prob >= THRESHOLD else (0, 200, 0)
            buffer = []

        # label + confidence at top left (below button)
        if label:
            cv2.putText(
                display,
                label,
                (20, 105),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.4,
                color,
                3,
                cv2.LINE_AA,
            )
            cv2.putText(
                display,
                f"Confidence: {confidence:.1f}%",
                (20, 132),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (200, 200, 200),
                1,
                cv2.LINE_AA,
            )

        # collecting indicator
        if len(buffer) > 0:
            cv2.putText(
                display,
                f"Collecting {len(buffer)}/{FRAME_COUNT}",
                (20, 155),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (150, 150, 150),
                1,
                cv2.LINE_AA,
            )

    # ── Draw button ───────────────────────────────────────────────────────────
    draw_button(display, analyzing)

    cv2.imshow("Violence Detection — Real Time", display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break
    if key == ord("a"):
        analyzing = not analyzing
        buffer = []
        if not analyzing:
            label = ""
            confidence = 0.0

cap.release()
cv2.destroyAllWindows()


Loading models...
Ready. Press A to toggle analysis. Press Q to quit.



In [1]:
import torch

if torch.cuda.is_available():
    print("CUDA is available!")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is NOT available.")
    print("Please ensure you have a compatible NVIDIA GPU and the CUDA Toolkit installed.")

CUDA is available!
CUDA Version: 12.4
Number of GPUs: 1
Current GPU Name: NVIDIA GeForce GTX 1650
